# Capstone 4 — Quiz master

A quiz is data plus a scoring policy. Questions live in a tuple of frozen records, so a running attempt cannot edit the paper. The engine accepts an answer, compares it case-insensitively, and can emit a marked report with the correct choice beside each miss.

The sample paper is short on purpose. Replacing `PAPER` is the only change required to run a different test.


In [1]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Question:
    prompt: str
    answer: str
    choices: tuple[str, ...]


PAPER = (
    Question("Which keyword starts a function?", "def", ("func", "def", "lambda", "fn")),
    Question("Which collection rejects duplicates?", "set", ("list", "tuple", "set", "dict")),
    Question("Which value means no value?", "None", ("0", "False", "None", "empty")),
    Question("Which statement leaves a loop?", "break", ("skip", "break", "pass", "stop")),
    Question("Which method adds to the end of a list?", "append", ("add", "push", "append", "insert")),
)


@dataclass
class Attempt:
    paper: tuple[Question, ...]
    responses: list[str]

    def __post_init__(self):
        if len(self.responses) != len(self.paper):
            raise ValueError("one response is required per question")

    def marked(self) -> list[tuple[Question, str, bool]]:
        rows = []
        for question, response in zip(self.paper, self.responses, strict=True):
            correct = response.strip().casefold() == question.answer.casefold()
            rows.append((question, response, correct))
        return rows

    @property
    def score(self) -> tuple[int, int]:
        earned = sum(1 for _, _, correct in self.marked() if correct)
        return earned, len(self.paper)

    def report(self) -> str:
        earned, total = self.score
        lines = [f"Score: {earned}/{total} ({earned / total:.0%})", ""]
        for number, (question, response, correct) in enumerate(self.marked(), start=1):
            mark = "correct" if correct else f"wrong, answer is {question.answer}"
            lines.append(f"{number}. {question.prompt}")
            lines.append(f"   you: {response}  ({mark})")
        return "\n".join(lines)


attempt = Attempt(PAPER, ["def", "list", "None", "break", "append"])
print(attempt.report())


Score: 4/5 (80%)

1. Which keyword starts a function?
   you: def  (correct)
2. Which collection rejects duplicates?
   you: list  (wrong, answer is set)
3. Which value means no value?
   you: None  (correct)
4. Which statement leaves a loop?
   you: break  (correct)
5. Which method adds to the end of a list?
   you: append  (correct)


## What to notice

Scoring is a property of the attempt, recomputed from the responses. There is no stored score to forget to update. `strict=True` on `zip` refuses a paper and a response list of different lengths, which keeps a partial submission from being marked as if it were complete.
